# **ML - Machine Learning**

## Objectives

- Develop a predictive model to identify students at risk of dropping out.
- Prioritise recall for dropout cases to reduce false negatives.
- Produce probability-based risk scores to support early intervention.

## Inputs

- Cleaned dataset: clean_student_dropout_dataset.csv
- Features (X): all variables except target
- Target (y): Dropout (1 = Dropout, 0 = Not Dropout)

## Outputs

- Trained Logistic Regression and Random Forest models
- Evaluation metrics: accuracy, precision, recall, F1, ROC-AUC
- Ranked dropout probabilities for students in the test set

---

## ML

In [27]:
# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score,
    accuracy_score,
    roc_auc_score,
    )

In [28]:
# Load the dataset and confirm target distribution
df = pd.read_csv("../Dataset/Processed/clean_student_dropout_dataset.csv")
print("Dataset shape:", df.shape)
print("Dropout distribution (0=No, 1=Yes):")
print(df["Dropout"].value_counts(normalize=True).rename("proportion"))
df.head()

Dataset shape: (9020, 19)
Dropout distribution (0=No, 1=Yes):
Dropout
0    0.765188
1    0.234812
Name: proportion, dtype: float64


,Student_ID,Age,Gender,Family_Income,Internet_Access,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Part_Time_Job,Scholarship,Stress_Index,GPA,Semester_GPA,CGPA,Semester,Department,Parental_Education,Dropout
0,1,22.1,0,25000.0,1,3.36,86.1,2,20.4,1,0,5.5,0.96,0.90,0.90,Year 1,Arts,High School,0
1,2,20.7,0,25000.0,1,4.30,68.0,2,44.0,0,0,6.8,1.28,1.20,1.19,Year 3,Engineering,Bachelor,1
2,3,22.4,0,40183.0,1,4.40,70.9,0,48.9,1,0,5.5,1.68,1.32,1.32,Year 1,Arts,Master,0
3,5,20.5,1,25319.0,1,4.19,75.7,1,23.0,0,0,7.0,1.48,0.91,0.87,Year 4,Business,Bachelor,0
4,7,24.5,0,25000.0,1,3.00,78.2,1,37.4,1,1,7.3,0.64,0.33,0.44,Year 4,CS,Bachelor,0


In [29]:
# Define features and target
X = pd.get_dummies(df.drop("Dropout", axis=1), drop_first=True)
y = df["Dropout"].astype(int)
print("Feature matrix shape:", X.shape)

Feature matrix shape: (9020, 25)


In [30]:
# Split the data into training and testing sets (stratified for class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train dropout rate:", round(y_train.mean(), 3))
print("Test dropout rate:", round(y_test.mean(), 3))

Train shape: (7216, 25)
Test shape: (1804, 25)
Train dropout rate: 0.235
Test dropout rate: 0.235


In [31]:
# Train a Logistic Regression model
model = LogisticRegression(
    max_iter=2000,
    random_state=42,
    class_weight="balanced",
    solver="liblinear",
)
model.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42,
                   solver='liblinear')

In [32]:
# Make predictions and probabilities for Logistic Regression
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

In [33]:
# Evaluate Logistic Regression
print("Logistic Regression - Classification Report:\n")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 3))

Logistic Regression - Classification Report:

              precision    recall  f1-score   support

           0       0.90      0.73      0.81      1380
           1       0.46      0.75      0.57       424

    accuracy                           0.74      1804
   macro avg       0.68      0.74      0.69      1804
weighted avg       0.80      0.74      0.75      1804

Confusion Matrix:
 [[1013  367]
 [ 107  317]]
ROC-AUC: 0.815


In [34]:
# Train a Random Forest Classifier
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced_subsample",
    min_samples_leaf=2,
    n_jobs=-1,
 )
rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced_subsample', min_samples_leaf=2,
                       n_estimators=300, n_jobs=-1, random_state=42)

In [35]:
# Compare models with a recall-first scorecard
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

comparison_df = pd.DataFrame(
    [
        {
            "Model": "Logistic Regression",
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision (dropout=1)": precision_score(y_test, y_pred, zero_division=0),
            "Recall (dropout=1)": recall_score(y_test, y_pred, zero_division=0),
            "F1 (dropout=1)": f1_score(y_test, y_pred, zero_division=0),
            "ROC-AUC": roc_auc_score(y_test, y_prob),
        },
        {
            "Model": "Random Forest",
            "Accuracy": accuracy_score(y_test, y_pred_rf),
            "Precision (dropout=1)": precision_score(y_test, y_pred_rf, zero_division=0),
            "Recall (dropout=1)": recall_score(y_test, y_pred_rf, zero_division=0),
            "F1 (dropout=1)": f1_score(y_test, y_pred_rf, zero_division=0),
            "ROC-AUC": roc_auc_score(y_test, y_prob_rf),
        },
    ]
).sort_values(["Recall (dropout=1)", "F1 (dropout=1)"], ascending=False)

print(comparison_df.round(3))

                 Model  Accuracy  Precision (dropout=1)  Recall (dropout=1)  \
0  Logistic Regression     0.737                  0.463               0.748   
1        Random Forest     0.798                  0.597               0.436   

   F1 (dropout=1)  ROC-AUC  
0           0.572    0.815  
1           0.504    0.796  


In [36]:
# Select the best model based on dropout recall (tie-breaker: F1)
best_model_name = comparison_df.iloc[0]["Model"]
best_model = model if best_model_name == "Logistic Regression" else rf
best_probs = y_prob if best_model_name == "Logistic Regression" else y_prob_rf

print("Best model by recall-first criterion:", best_model_name)

# Provide probability-based risk outputs required for intervention use
risk_results = X_test.copy()
risk_results["actual_dropout"] = y_test.values
risk_results["predicted_dropout"] = (best_probs >= 0.5).astype(int)
risk_results["dropout_probability"] = best_probs
print("Top 10 highest-risk students in test split:")
display(risk_results.sort_values("dropout_probability", ascending=False).head(10))

Best model by recall-first criterion: Logistic Regression
Top 10 highest-risk students in test split:


,Student_ID,Age,Gender,Family_Income,Internet_Access,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Part_Time_Job,...,Department_Business,Department_CS,Department_Engineering,Department_Science,Parental_Education_High School,Parental_Education_Master,Parental_Education_PhD,actual_dropout,predicted_dropout,dropout_probability
1959,2162,21.5,1,25000.0,0,5.85,72.1,3,41.6,1,...,False,False,True,False,False,False,False,1,1,0.972841
1901,2101,21.8,0,64457.0,0,4.93,58.8,4,34.6,0,...,False,True,False,False,True,False,False,1,1,0.969039
3230,3547,22.5,0,27426.0,0,3.75,74.7,3,33.5,1,...,False,False,False,True,False,False,False,0,1,0.967774
5013,5526,19.7,0,28083.0,1,3.37,73.9,1,41.1,1,...,False,False,True,False,False,False,False,1,1,0.963878
8916,9887,20.7,1,25000.0,0,2.33,69.8,5,47.8,1,...,False,False,False,False,False,True,False,1,1,0.960494
386,439,17.3,0,25000.0,1,3.73,51.5,4,20.6,0,...,False,False,False,False,True,False,False,1,1,0.958579
3698,4057,24.2,1,25000.0,1,5.72,77.5,4,25.1,0,...,True,False,False,False,False,False,False,1,1,0.957371
8649,9596,20.0,0,38470.0,1,3.90,72.6,4,29.3,1,...,False,False,False,False,False,True,False,0,1,0.957126
4661,5133,21.9,1,25000.0,0,3.73,76.5,0,19.0,1,...,False,False,True,False,False,False,False,1,1,0.954810
6751,7476,24.8,1,25000.0,0,3.38,76.2,1,26.0,0,...,False,False,True,False,True,False,False,1,1,0.954791


---

## Model Comparison: Logistic Regression vs Random Forest
Two models were evaluated for predicting student dropout using a stratified train-test split and class-imbalance-aware training. Because early intervention is the project priority, model ranking is recall-first for the dropout class (1), with F1-score as a tie-breaker.

### Logistic Regression

Pros:
- Simple and easy to interpret
- Works well with linearly separable data
- Fast to train and computationally efficient

Cons:
- Assumes a linear relationship between features and the target
- Struggles with complex, non-linear patterns

### Random Forest

Pros:
- Captures non-linear feature interactions
- Often improves minority-class detection when tuned
- Provides stable performance with mixed feature types

Cons:
- Less interpretable than Logistic Regression
- Computationally heavier
- Can still overfit without tuning and validation

### Final Model Selection
The selected model is the one with the highest recall for class 1 (dropout) on the test set. This criterion is intentional because missing an at-risk student (false negative) is costlier than flagging a student who later persists (false positive).

### Justification
The machine learning pipeline predicts student dropout using attendance, scholarship, and demographic-related inputs prepared from the cleaned dataset. A stratified split preserves the dropout ratio in both train and test sets, which improves the reliability of evaluation on imbalanced classes.

Both Logistic Regression and Random Forest were trained with class-imbalance handling. Model comparison includes accuracy, precision, recall, F1, and ROC-AUC, but final selection is recall-first for class 1 (dropout), consistent with the intervention objective.

The notebook also outputs dropout probabilities for each test observation so stakeholders can rank students by risk and prioritize support actions.

### Conclusion

The notebook produces a reproducible dropout-prediction workflow that runs top-to-bottom and evaluates two candidate classifiers under the same split.

Results show that attendance and financial-support-related variables remain important signals, and the selected model is chosen using a recall-first rule to maximize detection of at-risk students.

In addition to class predictions, the workflow generates dropout probabilities, enabling practical risk ranking for early intervention planning.

### Limitations
Despite stronger reproducibility and model evaluation, several limitations remain:

- Limited Feature Scope:
The dataset may omit important drivers of dropout, such as mental health, motivation, family responsibilities, or policy factors.
- Correlation vs Causation:
Predictive associations (for example with demographic variables) should not be interpreted as causal effects.
- Recall-Precision Trade-off:
Optimising for recall increases false positives, so intervention resources may be allocated to some students who would not drop out.
- Threshold Choice:
The current 0.5 decision threshold is a baseline; threshold tuning could improve the balance between recall and precision for operational use.
- No External Validation:
Results are based on a single train-test split. Cross-validation and testing on a separate cohort would improve confidence in generalisation.